# Day 4: Pandas Data Cleaning and Transformation
This notebook demonstrates basic data cleaning, handling missing values, type conversions, deduplication, derived columns, reshaping, and merging using Pandas, based on the Titanic `tested.csv` dataset.


In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
file_path = 'D:/Python notes/archive/tested.csv'
df = pd.read_csv(file_path)

# Display initial info
print("Initial Dataset Info:")
df.info()
display(df.head())


## Step 1: Identify and Handle Missing Values
**Reasoning:** 
Real-world data is often incomplete. We must handle missing values so they don't break our machine learning models or statistical calculations.
- **Age**: We will fill missing values with the median age. *Justification:* The median is robust to outliers, making it a safer imputation metric for continuous variables like age than the mean.
- **Fare**: We will fill missing values with the median fare for the same reason.
- **Cabin**: We will drop this column entirely. *Justification:* A massive portion (>75%) of the Cabin data is missing. Any imputation would be highly speculative and likely introduce more noise than value.


In [ ]:
# Identify missing values
print("Missing values before:\n", df.isnull().sum())

# Handle missing values
# 1. Fill 'Age' with median
df['Age'] = df['Age'].fillna(df['Age'].median())

# 2. Fill 'Fare' with median
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

# 3. Drop 'Cabin' column
if 'Cabin' in df.columns:
    df.drop('Cabin', axis=1, inplace=True)

# 4. Fill 'Embarked' with mode (if any missing)
if df['Embarked'].isnull().sum() > 0:
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

print("\nMissing values after:\n", df.isnull().sum())


## Step 2: Fix Wrong Data Types
**Reasoning:**
Storing categorical data (like passenger class or port of embarkation) as numbers or generic objects consumes more memory and doesn't explicitly convey their discrete nature to Pandas. Converting them to the `category` data type optimizes memory usage and allows for better semantic processing in future analysis.


In [ ]:
# Fix data types
df['Survived'] = df['Survived'].astype('category')
df['Pclass'] = df['Pclass'].astype('category')
df['Embarked'] = df['Embarked'].astype('category')

print("Data types after conversion:")
print(df.dtypes)


## Step 3: Remove Duplicates
**Reasoning:**
Duplicate rows can bias our dataset by giving certain observations disproportionate weight. It's standard practice to remove exact duplicates to ensure the integrity of the data distribution.


In [ ]:
# Check duplicates
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows found: {duplicate_count}")

# Remove duplicates
df.drop_duplicates(inplace=True)
print(f"Dataset shape after dropping duplicates: {df.shape}")


## Step 4: Create 2 New Derived Columns
**Reasoning:**
Feature engineering (creating new columns from existing ones) often exposes deeper insights than raw data. 
1. `FamilySize`: Knowing the total size of a passenger's family might correlate with survival better than isolated sibling/parent counts.
2. `FarePerPerson`: A grouped ticket might look exceptionally expensive, but calculating the fare *per person* gives us a truer understanding of the ticket's unit cost.


In [ ]:
# Create 'FamilySize' column
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Create 'FarePerPerson' column
df['FarePerPerson'] = df['Fare'] / df['FamilySize']

display(df[['Name', 'SibSp', 'Parch', 'FamilySize', 'Fare', 'FarePerPerson']].head())


## Step 5: Reshape the Data using `.groupby()` + `.agg()`
**Reasoning:**
Raw rows are hard to interpret at a macro level. Reshaping the data into an aggregated summary table allows us to immediately spot trends—for instance, how average age, fare, and passenger count differ across ticket classes and survival outcomes.


In [ ]:
# Reshaping data: Group by Pclass and Survived
summary_table = df.groupby(['Pclass', 'Survived'], observed=False).agg(
    Average_Age=('Age', 'mean'),
    Average_Fare=('Fare', 'mean'),
    Passenger_Count=('PassengerId', 'count')
).reset_index()

display(summary_table)


## Step 6: Merge with Manually Created DataFrame
**Reasoning:**
Codes like 'C', 'Q', and 'S' are not immediately readable to end-users or stakeholders. By mapping them to a manual contextual dataset, we enrich our main dataset, replacing obscure codes with their full, readable equivalents (Cherbourg, Queenstown, Southampton).


In [ ]:
# Create manual DataFrame for port mapping
port_mapping = pd.DataFrame({
    'Embarked': ['C', 'Q', 'S'],
    'PortName': ['Cherbourg', 'Queenstown', 'Southampton']
})

# Cast Embarked to category to match the main dataframe
port_mapping['Embarked'] = port_mapping['Embarked'].astype('category')

# Merge with the main DataFrame
df_merged = df.merge(port_mapping, on='Embarked', how='left')

# Show a sample of the merged data
display(df_merged[['PassengerId', 'Name', 'Embarked', 'PortName']].head(10))
